# Week 9 Exercise — MLflow Observability for a Simple Generative AI App

## Purpose

In this notebook, I recreate the MLflow observability example from the assigned source code. My goal is not only to make the application work, but also to explain what each major section is doing and why it matters.

The example combines a very small command-line style AI application with MLflow tracking. The application sends a conversation to the OpenAI API, receives a response, measures latency, counts tokens, and logs model settings and runtime metrics to MLflow.

I use **first-person documentation throughout the notebook** so that my reasoning, evaluation, and understanding of the code are clear.

## 1. Packages I Need

The assignment identifies three important packages:

- `mlflow` for experiment tracking and observability.
- `tiktoken` for token counting.
- `prometheus-client` as an optional monitoring-related dependency mentioned in the assignment.

The supplied example also imports `openai` and `colorama`, so I include those packages as well.

I would normally run the installation cell one time. If the packages are already installed in my environment, I do not need to reinstall them.

In [ ]:
# Run this cell once. If everything is already installed, pip will simply confirm it.
%pip install -q -U mlflow openai tiktoken prometheus-client colorama ipywidgets
print("Required packages are installed.")


## 2. Starting the MLflow Tracking Server

Before I run the application, I need an MLflow server running locally. The assignment specifies the following terminal command:

```bash
mlflow server
```

I run that command in a **separate terminal window** and leave the server running while I execute this notebook.

By default, this example expects MLflow at:

`http://localhost:5000`

I can open that address in a browser to inspect the experiment, runs, parameters, and metrics after the notebook logs data.

I intentionally do not start the server in a normal notebook cell because `mlflow server` is a long-running process and would keep that cell busy.

## 3. Imports and Configuration

In this section, I import the libraries used by the example and define the configuration values.

I keep the model configuration consistent with the supplied example because the purpose of the assignment is to recreate and evaluate that example rather than redesign it.

I also keep the API key outside the notebook. I do **not** hard-code a secret key into my submission. Instead, I read it from an environment variable named `OPENAI_API_BOOK_KEY`.

In [ ]:
import os
import time
import warnings
from getpass import getpass
from urllib.request import urlopen
from urllib.error import URLError

import mlflow
from openai import OpenAI
import tiktoken as tk
from colorama import Fore, Style, init

# Hide the harmless tqdm/Jupyter progress-bar warning if it appears.
warnings.filterwarnings(
    "ignore",
    message=".*IProgress not found.*"
)

# -----------------------------
# Configuration
# -----------------------------
MLFLOW_URI = "http://localhost:5000"

# MODEL MUST be defined before it is printed or used.
# This is a current small OpenAI chat model. If your account does not have access,
# replace only this value with a model available to your API account.
MODEL = "gpt-4o-mini"

TEMPERATURE = 0.7
TOP_P = 1
FREQUENCY_PENALTY = 0
PRESENCE_PENALTY = 0
MAX_TOKENS = 300
DEBUG = False

# Accept the variable name used by the original assignment notebook and
# the standard OpenAI environment-variable name.
API_KEY = (
    os.getenv("OPENAI_API_BOOK_KEY")
    or os.getenv("OPENAI_API_KEY")
)

# If a key is not already stored in the environment, ask for it securely.
# getpass hides what you type, so the key is not displayed in the notebook output.
if not API_KEY:
    API_KEY = getpass(
        "Paste your OpenAI API key here (input will be hidden), then press Enter: "
    ).strip()

print("Configuration loaded.")
print("MLflow URI:", MLFLOW_URI)
print("Model:", MODEL)
print("API key found:", bool(API_KEY))


### My evaluation of the configuration

I think keeping configuration values together near the top of the notebook is a good design choice because it makes the application easier to understand and modify. Instead of searching through multiple functions, I can see the model, token limit, sampling settings, and tracking location in one place.

The most important security choice is the API key. I use an environment variable because putting a real API key directly into a notebook could expose it when the notebook is submitted, shared, or uploaded to GitHub.

## 4. Initialize Color Output, the OpenAI Client, and MLflow

The original example initializes `colorama`, creates an OpenAI client, points MLflow to the local server, and selects an experiment named `GenAI_book`.

MLflow experiments are useful because they organize related runs. Each time I start a new run, MLflow can store the parameters and metrics associated with that run.

In [ ]:
# Initialize terminal-style colors.
init()

# Create the OpenAI client.
if not API_KEY:
    raise RuntimeError(
        "No OpenAI API key was provided. Rerun the configuration cell and enter your key."
    )

client = OpenAI(api_key=API_KEY)

# Configure MLflow.
mlflow.set_tracking_uri(MLFLOW_URI)

# Check whether the local MLflow server is reachable before continuing.
try:
    with urlopen(MLFLOW_URI, timeout=3) as response:
        server_status = response.status
    print(f"MLflow server is reachable (HTTP {server_status}).")
except Exception:
    raise RuntimeError(
        "MLflow is not reachable at http://localhost:5000.\n\n"
        "Open Anaconda Prompt / Command Prompt / PowerShell in a SEPARATE window and run:\n"
        "    mlflow server\n\n"
        "Leave that window open, then rerun this cell."
    )

# Create or select the experiment.
mlflow.set_experiment("GenAI_book")
print("MLflow tracking is configured.")


### What I expect to see in MLflow

After the application runs successfully, I expect the MLflow interface to contain a run under the `GenAI_book` experiment.

The run should include model parameters such as temperature and `top_p`, along with runtime metrics such as request latency and token counts. This is the core observability feature demonstrated by the example: I can inspect what configuration produced a response and how expensive or slow that request was.

## 5. Helper Functions for Display

The supplied example uses two small helper functions to make user input and AI output easier to distinguish visually.

These functions do not affect the model itself. They only improve readability for the person using the application.

In [ ]:
def print_user_input(text):
    """Display user text clearly."""
    print(f"{Fore.GREEN}You:{Style.RESET_ALL}", text)


def print_ai_output(text):
    """Display assistant text clearly."""
    print(f"{Fore.BLUE}AI Assistant:{Style.RESET_ALL}", text)


## 6. Token Counting Function

The example uses `tiktoken` to estimate the number of tokens in a string.

Tokens matter because language models do not process text strictly as words. A token may be a word, part of a word, punctuation, or another small text unit. Tracking token counts helps me understand the size of the prompt and response.

The function below gets the `cl100k_base` tokenizer, encodes the text, and returns the number of encoded tokens.

In [ ]:
def count_tokens(text, encoding_name="cl100k_base") -> int:
    """Count approximate tokens in text with tiktoken."""
    if text is None:
        return 0

    encoding = tk.get_encoding(encoding_name)
    encoded = encoding.encode(str(text), disallowed_special=())
    return len(encoded)


### My evaluation of token counting

I think token counting is one of the most useful parts of this example because it gives the MLflow run more context than simply recording whether the request succeeded.

The example logs three token-related values:

- the most recent user prompt,
- the full conversation converted to a string,
- the completion returned by the model.

That gives me a simple way to compare how the conversation grows over time and how large each response is.

One limitation is that this is a manual estimate based on locally encoding strings. A production system could also record token usage reported directly by the model API when available. However, for this exercise, the local counter clearly demonstrates the monitoring idea.

## 7. Main Text-Generation Function

The `generate_text()` function is the most important function in the notebook.

I use it to perform five main tasks:

1. record the request start time,
2. send the conversation to the OpenAI model,
3. measure latency,
4. count prompt/conversation/completion tokens,
5. log metrics and model parameters to MLflow.

This function shows how application behavior and observability can be combined in one workflow.

In [ ]:
def generate_text(conversation, max_tokens=MAX_TOKENS) -> str:
    """
    Send a conversation to OpenAI and log model settings and runtime metrics to MLflow.
    """

    if not conversation:
        raise ValueError("conversation cannot be empty.")

    start_time = time.perf_counter()

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=conversation,
            temperature=TEMPERATURE,
            max_tokens=max_tokens,
            top_p=TOP_P,
            frequency_penalty=FREQUENCY_PENALTY,
            presence_penalty=PRESENCE_PENALTY
        )
    except Exception as exc:
        # Log the failure when a run is active, then show a clean error.
        if mlflow.active_run() is not None:
            mlflow.log_metric("request_error", 1)
        raise RuntimeError(
            "The OpenAI request failed. Check that your API key is valid, "
            "your account has API billing/credits, and the selected model is available.\n"
            f"Original error: {exc}"
        ) from exc

    latency = time.perf_counter() - start_time

    message_response = response.choices[0].message.content or ""

    # Prefer token counts returned by the API when available.
    usage = getattr(response, "usage", None)
    if usage is not None:
        prompt_tokens = int(getattr(usage, "prompt_tokens", 0) or 0)
        completion_tokens = int(getattr(usage, "completion_tokens", 0) or 0)
        total_tokens = int(getattr(usage, "total_tokens", 0) or 0)
    else:
        prompt_tokens = count_tokens(conversation[-1].get("content", ""))
        completion_tokens = count_tokens(message_response)
        total_tokens = prompt_tokens + completion_tokens

    conversation_tokens = count_tokens(
        " ".join(str(item.get("content", "")) for item in conversation)
    )

    # Log metrics for this request.
    mlflow.log_metrics({
        "request_count": 1,
        "request_latency_seconds": float(latency),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "conversation_tokens": conversation_tokens,
        "request_error": 0
    })

    if DEBUG:
        active = mlflow.active_run()
        if active:
            print("Run ID:", active.info.run_id)

    return message_response


### Why metrics and parameters are separated

I understand MLflow parameters as values that describe **how I configured the model**, while metrics describe **what happened during execution**.

For example, `temperature` is a parameter because I choose it before the request. `request_latency` is a metric because I only know it after the request completes.

This separation makes the MLflow run easier to analyze. I can ask questions such as whether a certain configuration corresponds with larger completions, higher latency, or other behavior.

## 8. Quick Local Test of the Token Counter

Before making a paid or authenticated API request, I can test the local token-counting function independently.

This confirms that `tiktoken` is installed and that the helper function works.

In [ ]:
sample_text = "MLflow helps me track model experiments and runtime behavior."
sample_count = count_tokens(sample_text)

print("Sample text:", sample_text)
print("Token count:", sample_count)
print("Token counter test passed.")


## 9. Run the Simple Conversation App

The original Python file uses an infinite `while` loop and `input()` so the user can continue chatting until entering `exit`, `quit`, `q`, or `e`.

I can use the same approach in Jupyter Notebook. The notebook will wait for my input each time the loop reaches `input()`.

Before I run this cell, I make sure:

- the MLflow server is running,
- my API key environment variable is set,
- my internet connection allows the API request.

I also call `mlflow.autolog()` as in the supplied example and start one explicit MLflow run around the conversation.

In [ ]:
# Simple Jupyter chat application.
# Make sure the MLflow server is still running before executing this cell.

conversation = [
    {"role": "system", "content": "You are a helpful assistant."}
]

with mlflow.start_run(run_name="week9_mlflow_chat") as run:

    # Parameters describe how the model is configured, so I log them once per run.
    mlflow.log_params({
        "model": MODEL,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "frequency_penalty": FREQUENCY_PENALTY,
        "presence_penalty": PRESENCE_PENALTY,
        "max_tokens": MAX_TOKENS
    })

    print("MLflow Run ID:", run.info.run_id)
    print("Type exit, quit, q, or e to stop.\n")

    while True:
        user_input = input("User: ").strip()

        if user_input.lower() in {"exit", "quit", "q", "e"}:
            print("Conversation ended.")
            break

        if not user_input:
            print("Please type a message.")
            continue

        conversation.append({
            "role": "user",
            "content": user_input
        })

        try:
            ai_output = generate_text(conversation)
        except Exception as exc:
            print("\nERROR:")
            print(exc)
            print("\nThe MLflow run will stop so you can correct the problem and rerun this cell.")
            break

        print_ai_output(ai_output)

        conversation.append({
            "role": "assistant",
            "content": ai_output
        })

print("\nOpen http://localhost:5000 and select the GenAI_book experiment to view the run.")


## 10. What I Would Inspect in the MLflow Interface

After I complete at least one prompt, I open the local MLflow interface at `http://localhost:5000`.

I select the `GenAI_book` experiment and inspect the run created by this notebook.

I expect to review the following information:

**Parameters**
- model
- temperature
- top_p
- frequency penalty
- presence penalty

**Metrics**
- request count
- request latency
- prompt tokens
- completion tokens
- conversation tokens

I would confirm that the values shown in the MLflow interface match the values defined or calculated in this notebook.

If I submit screenshots with the notebook, I would include a screenshot of the experiment/run page only if my instructor allows or requests it. The notebook itself already contains the required explanation.

## 11. My Commentary and Evaluation

The main lesson I take from this exercise is that observability should be designed into an AI application rather than treated as an afterthought.

The actual chat application is very small. It stores a conversation, sends it to a model, prints the response, and repeats. MLflow adds a second layer that records information about how the application behaved.

I see several advantages in this design. First, I can reproduce or compare experiments because the model configuration is recorded. Second, I can measure performance through request latency. Third, I can monitor the approximate size of inputs and outputs through token counts. If I were comparing multiple runs, these values could help me identify whether changes in configuration are associated with slower responses or larger completions.

I also noticed that this example is intentionally simple. It logs `request_count` as `1` for each call, so it is not a complete production monitoring system. It also does not log user satisfaction, response quality, exceptions, cost estimates, or safety-related outcomes. Those would be useful additions in a larger application.

Another limitation is that the Prometheus client is mentioned but not actively used in the supplied code. I interpret that as evidence that the exercise is primarily focused on MLflow tracking rather than building a full monitoring stack.

Overall, I think the example succeeds because it makes observability concrete. Instead of discussing monitoring only in theory, I can see exactly where the application measures latency, counts tokens, records model settings, and sends that information to MLflow.

## Important Troubleshooting Note

This corrected notebook defines `MODEL` in the configuration cell **before** it is printed or used.  
If Jupyter ever reports that a variable such as `MODEL`, `API_KEY`, or `MLFLOW_URI` is not defined, use **Kernel → Restart Kernel and Run All Cells** so the notebook executes in order from the beginning.

The notebook also asks for the OpenAI API key securely with `getpass()` when it is not already stored as an environment variable. The key is hidden while it is entered and is not written into the saved notebook.



## 12. Troubleshooting Notes

I documented several problems I might encounter so that the notebook is easier to reproduce.

**If MLflow cannot connect:**  
I check that `mlflow server` is still running and that the notebook uses `http://localhost:5000`.

**If the OpenAI request fails immediately:**  
I check whether `OPENAI_API_BOOK_KEY` is defined in the environment and whether the key is valid.

**If an import fails:**  
I rerun the installation cell and restart the Jupyter kernel if necessary.

**If the selected model is unavailable in my account/environment:**  
I would document the error rather than silently changing the assignment example. If I needed to substitute a model to complete execution, I would clearly state that substitution in my notebook so that my work remains transparent.

**If no run appears in MLflow:**  
I verify that `mlflow.set_tracking_uri()` points to the same server I started and confirm that the conversation cell actually entered an MLflow run.

## 13. Conclusion

In this notebook, I recreated the supplied MLflow observability example as a Jupyter Notebook and documented each important step in my own words.

I installed the required packages, configured the OpenAI and MLflow clients, created token-counting and display helpers, built the text-generation function, measured latency, logged metrics and parameters, and adapted the simple interactive application so it can run inside Jupyter.

The most valuable part of the exercise for me is seeing how a small amount of monitoring code can make an AI application much easier to inspect. MLflow gives me a structured record of the model configuration and the runtime behavior of the requests, which is important for debugging, comparison, and responsible experimentation.